# Examen Segundo Parcial

## Nombre: Pedro Jhoel Antonio Magne Ordoñez

## Ejercicio 16.   Ahorcado simple: Diseña un agente que adivine palabras cortas. Explica cómo explora letras posibles y cómo explota letras frecuentes o pistas conocidas.

In [1]:
import numpy as np
from tqdm import tqdm

### El entorno que hay es el juego de la clase ahorcado y la clase game, donde el agente interacciona con estas clases para aprender a resolver el juego

### Las recompensas que el agente obtiene son:
- Letra correcta que completa palabra: +20.0 y terminado=True.
- Letra correcta no final: +2.0, terminado=False.
- Letra incorrecta: -1.0, vidas--.
- Si vidas==0: -20.0 y terminado=True.


### Agente usado: Selección de acciones con intervalo de confianza


$$
    A_t = \underset{a}{\arg\max} \, \left[ Q_t(a) + c \sqrt{\frac{\text{ln} \, t}{N_t(a)}} \right]
$$

Políticas: 
- UCB (Agent.move con explore=True): Selecciona letra por q_val + c * sqrt(log(global_steps)/n_val); 
- Explotación (Agent.move con explore=False): Elige la letra con mayor q_val.
- Histórico episódico (Agent.history) usado para actualizar al final.

#### En este agente la función de valor es Q(s,a): asigna a cada par (estado, acción) la expectativa del retorno acumulado. Se almacena en q_table y se actualiza al final del episodio.
#### Durante la exploración UCB se combina Q(s,a) con el término de exploración c·sqrt(log t / N(s,a)).

In [2]:
class Ahorcado:
    def __init__(self):
        self.vocabulario = ["gato", "casa", "pex", "carro", "silla", "perro", "toma", "summer"]
        self.reset()

    def reset(self):
        self.palabraEscogida = np.random.choice(self.vocabulario)
        self.vidas = 6
        self.letras_usadas = set()
        return self.get_state()

    def get_state(self):
        return "".join([letra if letra in self.letras_usadas else "_" for letra in self.palabraEscogida])

    def valid_moves(self):
        # abcdefghijklmnopqrstuvwxyz
        abecedario = list("abcdefghijklmnopqrstuvwxyz")
        return [letra for letra in abecedario if letra not in self.letras_usadas]

    def step(self, letra):
        self.letras_usadas.add(letra)
        if letra in self.palabraEscogida:
            estado_actual = self.get_state()
            if estado_actual == self.palabraEscogida:
                return estado_actual, 20.0, True
            return estado_actual, 2.0, False
        else:
            self.vidas -= 1
            estado_actual = self.get_state()
            if self.vidas == 0:
                return estado_actual, -20.0, True
            return estado_actual, -1.0, False

In [3]:
class Agent:
    def __init__(self, alpha=0.1, c=1.5):
        self.q_table = {}      
        
        # cuántas veces se eligio cada acción en cada estado
        self.n_table = {}          
        self.global_steps = 0      
        self.alpha = alpha
        self.c = c                 
        self.history = []          

    def reset(self):
        self.history = []

    def _get_q_n(self, state_str, action):
        if state_str not in self.q_table:
            self.q_table[state_str] = {}
            self.n_table[state_str] = {}
        if action not in self.q_table[state_str]:
            self.q_table[state_str][action] = 0.0
            self.n_table[state_str][action] = 0
        return self.q_table[state_str][action], self.n_table[state_str][action]

    def move(self, env, explore=True):
        state_str = env.get_state()
        valid_moves = env.valid_moves()
        
        best_letter = None
        best_ucb_value = -float('inf')
        
        for letra in valid_moves:
            q_val, n_val = self._get_q_n(state_str, letra)
            
            # Exploracion

            if explore:
                self.global_steps += 1
               

                if n_val == 0:
                    ucb_val = float('inf')
                else:
                    # Fórmula de UCB
                    ucb_val = q_val + self.c * np.sqrt(np.log(self.global_steps) / n_val)
            else:
                # Explotacion
                ucb_val = q_val
                
            if ucb_val > best_ucb_value:
                best_ucb_value = ucb_val
                best_letter = letra
                
        if explore:
            self.history.append((state_str, best_letter))
            
        return best_letter

    def reward(self, reward):
        for state_str, action in reversed(self.history):
            q_val = self.q_table[state_str][action]
            
            # al final de la partida (cuando recibimos la recompensa)
            # iteramos por tods los estados actualizando su valor en la tabla
            self.n_table[state_str][action] += 1
            
            # Ajuste de la tabla de valores
            target = reward
            self.q_table[state_str][action] += self.alpha * (target - q_val)
            
            reward -= 0.5

In [4]:
class Game:
    def __init__(self, board, player):
        self.board = board
        self.player = player

    def selfplay(self, explore=True):
        state = self.board.reset()
        self.player.reset()
        terminado = False
        total_reward = 0
        
        while not terminado:
            letra = self.player.move(self.board, explore=explore)
            state, reward, terminado = self.board.step(letra)
            total_reward += reward
            
        if explore:
            self.player.reward(total_reward)
            
        return state == self.board.palabraEscogida

    def entrenamiento(self, num_episodes=5000):
        print(f"Entrenando.... ({num_episodes} partidas)...")
        victorias = 0
        
        for i in range(1, num_episodes + 1):
            won = self.selfplay(explore=True)
            if won:
                victorias += 1
                
            if i % 10 == 0:
                tasa_victorias = (victorias / 1000) * 100
                print(f"Partidas {i-999}-{i} -> Tasa de Éxito UCB: {tasa_victorias:.1f}%")
                victorias = 0

In [5]:
# Entrenamiento
juego = Ahorcado()
agenteUCB = Agent(alpha=0.1, c=1.5)
game = Game(juego, agenteUCB)


game.entrenamiento(num_episodes=50000)


Entrenando.... (50000 partidas)...
Partidas -989-10 -> Tasa de Éxito UCB: 0.0%
Partidas -979-20 -> Tasa de Éxito UCB: 0.1%
Partidas -969-30 -> Tasa de Éxito UCB: 0.1%
Partidas -959-40 -> Tasa de Éxito UCB: 0.0%
Partidas -949-50 -> Tasa de Éxito UCB: 0.0%
Partidas -939-60 -> Tasa de Éxito UCB: 0.0%
Partidas -929-70 -> Tasa de Éxito UCB: 0.0%
Partidas -919-80 -> Tasa de Éxito UCB: 0.0%
Partidas -909-90 -> Tasa de Éxito UCB: 0.0%
Partidas -899-100 -> Tasa de Éxito UCB: 0.0%
Partidas -889-110 -> Tasa de Éxito UCB: 0.0%
Partidas -879-120 -> Tasa de Éxito UCB: 0.0%
Partidas -869-130 -> Tasa de Éxito UCB: 0.0%
Partidas -859-140 -> Tasa de Éxito UCB: 0.0%
Partidas -849-150 -> Tasa de Éxito UCB: 0.0%
Partidas -839-160 -> Tasa de Éxito UCB: 0.0%
Partidas -829-170 -> Tasa de Éxito UCB: 0.0%
Partidas -819-180 -> Tasa de Éxito UCB: 0.0%
Partidas -809-190 -> Tasa de Éxito UCB: 0.1%
Partidas -799-200 -> Tasa de Éxito UCB: 0.0%
Partidas -789-210 -> Tasa de Éxito UCB: 0.0%
Partidas -779-220 -> Tasa de 